<a href="https://colab.research.google.com/github/tomo-mar/project_research_A/blob/main/Highlight_ZeroShot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ディレクトリ設定

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/VAD_Experiment"
INPUT_DIR = os.path.join(BASE_DIR, "input")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")

os.makedirs(OUTPUT_DIR, exist_ok=True)

プロンプトの定義

In [ ]:
# 1.直接ハイライト判定のプロンプト
PROMPT_1_DIRECT = """
あなたは優秀な動画解析アシスタントです。
入力された5秒間の音声クリップを聞き、配信のハイライト（盛り上がり）確率を判定してください。

【評価基準】
0.0に近いほどベースライン（通常のトーン、特筆すべき起伏なし）
1.0に近いほどピーク（絶叫、爆笑など、最高潮の瞬間）

【出力形式】
判定結果となる「0.0 から 1.0 の間の数値（小数点以下2桁）」のみを出力してください。
例: 0.85
"""

# 2.VAD経由でのハイライト判定（Zero-shot CoT）のプロンプト
PROMPT_2_VAD_COT = """
あなたは優秀な動画解析アシスタントです。
入力された5秒間の音声クリップを聞き、配信のハイライト確率を判定してください。
直感で判定するのではなく、必ず音声から「VAD（感情の3次元モデル）」を推測し、そのVADからどのように確率を推定したかのプロセスを経由して最終判定を下してください。

【VADの定義】
- Valence (快・不快): 配信者の感情がポジティブかネガティブか
- Arousal (興奮度): 配信者の覚醒度、エネルギーの高さ
- Dominance (支配度): 状況をコントロールしているか、圧倒されているか

【出力形式】
必ず以下のフォーマットに従って出力してください。

1. VADスコアの推測
- Valence: [低/中/高]
- Arousal: [低/中/高]
- Dominance: [低/中/高]

2. ハイライト確率の推定理由
[推測した上記のVADスコアの組み合わせから、なぜそのハイライト確率になるのか、思考プロセスを簡潔に記述]

3. 最終判定
[最終的なハイライト確率を 0.0 から 1.0 の間の数値で記述]
"""
print("プロンプトの定義が完了")

モデルと推論関数の定義

In [ ]:
!pip install -q pydub librosa
!pip install -q git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install -q qwen-omni-utils

import torch
import librosa
import re
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
import logging
logging.getLogger().setLevel(logging.ERROR)

print("モデルをロード")
model_path = "Qwen/Qwen2.5-Omni-7B"

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_path)
print("モデルのロードが完了")

def run_qwen_inference(filepath, prompt):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are a highly capable audio and video analysis AI."}]},
        {"role": "user", "content": [
            {"type": "audio", "audio_url": filepath},
            {"type": "text", "text": prompt}
        ]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    audio_array, _ = librosa.load(filepath, sr=16000)
    inputs = processor(text=text, audio=[audio_array], return_tensors="pt", padding=True).to("cuda")

    generate_ids = model.generate(**inputs, max_new_tokens=150, return_audio=False)
    generate_ids = [cur_ids[len(inputs.input_ids[0]):] for cur_ids in generate_ids]
    output_text = processor.batch_decode(generate_ids, skip_special_tokens=True)[0]

    return output_text

def extract_probability(text):
    matches = re.findall(r"0\.\d+|1\.0+", text)
    if matches:
        return float(matches[-1])
    return 0.0

print("推論ラッパー関数の定義が完了")

メイン処理

In [ ]:
import pandas as pd
from pydub import AudioSegment

def process_audio_files():
    wav_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.wav')]

    if not wav_files:
        print(f"{INPUT_DIR} にwavファイルが存在しない")
        return

    for filename in wav_files:
        print(f"\n処理開始: {filename}")
        input_path = os.path.join(INPUT_DIR, filename)

        audio = AudioSegment.from_wav(input_path)
        results = []
        chunk_length_ms = 5000

        total_chunks = sum(1 for s in range(0, len(audio), chunk_length_ms) if len(audio[s:min(s + chunk_length_ms, len(audio))]) >= 1000)
        current_chunk = 0

        for start_ms in range(0, len(audio), chunk_length_ms):
            end_ms = min(start_ms + chunk_length_ms, len(audio))
            chunk = audio[start_ms:end_ms]

            if len(chunk) < 1000:
                continue

            current_chunk += 1
            time_sec = start_ms / 1000.0

            # print文を進捗率付きに変更
            print(f"  [{current_chunk}/{total_chunks}]を解析中")

            temp_chunk_path = "/content/temp_chunk.wav"
            chunk.export(temp_chunk_path, format="wav")

            # 1.直接推論
            res1_text = run_qwen_inference(temp_chunk_path, PROMPT_1_DIRECT)
            score1 = extract_probability(res1_text)

            # 2.VAD経由推論
            res2_text = run_qwen_inference(temp_chunk_path, PROMPT_2_VAD_COT)
            score2 = extract_probability(res2_text)

            results.append({
                "Time_sec": time_sec,
                "Score_Direct": score1,
                "Score_VAD": score2,
                "RawText_Direct": res1_text,
                "Reason_VAD": res2_text
            })

        # CSVへ保存
        df = pd.DataFrame(results)
        output_csv_name = filename.replace(".wav", ".csv")
        output_csv_path = os.path.join(OUTPUT_DIR, output_csv_name)
        df.to_csv(output_csv_path, index=False)
        print(f"保存完了: {output_csv_path}")

# 実行
process_audio_files()
print("\n処理完了")